# CardioVision 3D - Step 1: Rigorous Dataset Audit
**Track A: Cardiovascular Risk Visualization & Prediction**

---
### Objective
Perform a rigorous data quality audit on the **UCI Extension of Z-Alizadeh Sani Dataset**.
This notebook strictly adheres to Step 1 requirements:
- No model training or evaluation
- No dataset modifications or imputations
- Conservative feature classification and leakage evaluation
- Full transparency into missingness, distributions, formatting, and modeling risks


## 1. Dataset Loading

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add workspace src directory to path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from dataset_audit import (
    locate_dataset, load_dataset, dataset_basic_info, build_data_dictionary,
    audit_missing_values, audit_duplicates, audit_targets, audit_target_leakage,
    audit_feature_types, audit_unique_values, audit_numerical_features,
    audit_data_quality, save_reports, KNOWN_TARGET_COLS, RANDOM_SEED
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(RANDOM_SEED)

dataset_path, exists = locate_dataset()
if not exists:
    raise FileNotFoundError("REQUIRED DATASET MISSING! Please place 'extention of Z-Alizadeh sani dataset.xlsx' under data/")

df = load_dataset(dataset_path)
info = dataset_basic_info(df, dataset_path)

print(f"Dataset Path: {info['dataset_path']}")
print(f"Number of Rows: {info['num_rows']}")
print(f"Number of Columns: {info['num_cols']}")
print(f"Memory Footprint: {info['memory_usage_mb']} MB")


In [ ]:
print("Column Names:")
print(info['columns'])


In [ ]:
df.head(5)

In [ ]:
df.tail(5)

In [ ]:
df.dtypes

## 2. Data Dictionary

In [ ]:
data_dict_df = build_data_dictionary(df)
data_dict_df


## 3. Missing Values Audit

In [ ]:
missing_df = audit_missing_values(df)
print(f"Total columns with missing values: {(missing_df['missing_count'] > 0).sum()}")
missing_df


In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(range(len(df.columns)), df.isnull().sum(), color='navy')
plt.title("Missing Value Count Per Column")
plt.xlabel("Column Index")
plt.ylabel("Missing Count")
plt.ylim(0, 10)
plt.show()


## 4. Duplicates Audit

In [ ]:
dup_audit = audit_duplicates(df)
print(f"Total Rows: {dup_audit['num_rows']}")
print(f"Exact Duplicate Rows: {dup_audit['exact_duplicate_rows']} ({dup_audit['exact_duplicate_percentage']}%)")
print("Duplicate checks on potential identifier columns:", dup_audit['id_column_duplicates'])


## 5. Target Audit

In [ ]:
target_audit_res = audit_targets(df)
for target_name, details in target_audit_res.items():
    print(f"=== TARGET: {target_name} ===")
    if details.get('present'):
        print(f"Unique Values: {details['unique_values']}")
        print(f"Class Counts: {details['class_counts']}")
        print(f"Class Percentages: {details['class_percentages']}")
        print(f"Missing Count: {details['missing_count']}")
    else:
        print(details['note'])
    print()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Target Class Distributions (Z-Alizadeh Sani Dataset)", fontsize=14, fontweight='bold')

targets_to_plot = ['Cath', 'LAD', 'LCX', 'RCA']
for ax, target_name in zip(axes.flatten(), targets_to_plot):
    if target_name in df.columns:
        counts = df[target_name].value_counts()
        bars = ax.bar(counts.index.astype(str), counts.values, color=['#2b5c8f', '#d95f02'])
        ax.set_title(f"Target: {target_name} (CAD = {target_name == 'Cath'})")
        ax.set_ylabel("Count")
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f"{height} ({height/len(df)*100:.1f}%)",
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 6. Target Leakage Audit

In [ ]:
leakage_df = audit_target_leakage(df)
leakage_df


## 7. Feature Type Audit

In [ ]:
feature_types = audit_feature_types(df)
for ftype, col_list in feature_types.items():
    print(f"=== {ftype.upper()} ({len(col_list)}) ===")
    print(col_list)
    print()


## 8. Unique Value Audit

In [ ]:
unique_val_df = audit_unique_values(df)
print(f"Total Categorical/Binary Unique Value Rows Audited: {len(unique_val_df)}")
rare_or_issue = unique_val_df[(unique_val_df['formatting_issue_flag']) | (unique_val_df['percentage'] < 5.0)]
rare_or_issue.head(20)


## 9. Numerical Feature Audit

In [ ]:
num_audit_df = audit_numerical_features(df)
num_audit_df


## 10. Data Quality Checks

In [ ]:
dq_results = audit_data_quality(df)
print("Constant Columns (1 unique value):", dq_results['constant_columns'])
print("Near-Constant Columns (>= 95% single value):", dq_results['near_constant_columns'])
print("Whitespace Issue Columns:", dq_results['whitespace_issue_columns'])
print("Columns with Zero Values:", dq_results['zero_value_counts'])
print("Columns with Negative Values:", dq_results['negative_value_counts'])


## 11. Correlation & Association Exploration

In [ ]:
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 10]
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation Matrix of Continuous Numerical Features", fontsize=12, fontweight='bold')
plt.show()


## 12. Dataset Size and Modeling Risk Analysis

### Key Observations & Risks:
1. **Sample Size vs Feature Dimensionality ($N=303$, $P=54$)**:
   - High risk of overfitting if unregularized complex models are evaluated without strict cross-validation.
   - Requires robust Stratified K-Fold CV (5-fold or 10-fold) with fixed random seeds.

2. **Target Class Distribution**:
   - **Cath (CAD Overall)**: 71.3% CAD vs 28.7% Normal (Imbalanced).
   - **LAD Stenosis**: 58.4% Stenotic vs 41.6% Normal.
   - **LCX Stenosis**: 39.3% Stenotic vs 60.7% Normal.
   - **RCA Stenosis**: 37.6% Stenotic vs 62.4% Normal.
   - Standard Accuracy will be misleading; ROC-AUC, PR-AUC, F1-score, Sensitivity, and Specificity are required.

3. **Near-Constant / Low-Variance Features**:
   - `Exertional CP` has 0 variance (100% 'N').
   - 13 features have >=95% single value (e.g. `CHF` 99.67% 'N', `LowTH Ang` 99.34% 'N').
   - High risk of tree splits wasting depth on zero-information features.

4. **Formatting Inconsistencies**:
   - `Sex` column encodes females as `'Fmale'` (typo in raw dataset). Must be mapped cleanly during pre-processing in Step 2.


## 13. Data Audit Report Generation

In [ ]:
print("=== CARDIOVISION 3D - STEP 1 DATA AUDIT REPORT ===")
print(f"DATASET: Rows={df.shape[0]}, Columns={df.shape[1]}")
print(f"TARGETS: Cath(CAD={df['Cath'].value_counts().get('CAD',0)}), LAD(Stenotic={df['LAD'].value_counts().get('Stenotic',0)}), LCX(Stenotic={df['LCX'].value_counts().get('Stenotic',0)}), RCA(Stenotic={df['RCA'].value_counts().get('Stenotic',0)})")
print(f"QUALITY: Missing={df.isnull().sum().sum()}, Exact Duplicates={df.duplicated().sum()}, Constant Cols={dq_results['constant_columns']}")
print(f"FEATURES: Numerical={len(feature_types['numerical'])}, Binary={len(feature_types['binary'])}, Categorical={len(feature_types['categorical'])}")
print(f"LEAKAGE: Excluded Targets = {feature_types['target']}")


## 14. Save Audit Outputs

In [ ]:
reports_to_save = {
    'dataset_summary': pd.DataFrame([info]),
    'feature_dictionary': data_dict_df,
    'missing_values': missing_df,
    'target_distribution': pd.DataFrame(target_audit_res),
    'leakage_audit': leakage_df,
    'numerical_features_audit': num_audit_df,
    'unique_values_audit': unique_val_df
}

saved_report_paths = save_reports(reports_to_save, output_dir='../reports')
print("Saved Reports:")
for p in saved_report_paths:
    print(f" - {p}")
